## Observing the CMB

In [ ]:
import maria
from maria.band import get_band

f090 = get_band("act/pa5/f090")
f150 = get_band("act/pa5/f150")

f090.NET_RJ = 50e-6
f150.NET_RJ = 50e-6

f090.knee = 1e2
f150.knee = 1e2

array = {"field_of_view": 0.3,
         "primary_size": 10,
         "n": 100,
         "packing": "sunflower",
         "shape": "circle",
         "polarized": True,
         "bands": [f090, f150]}

instrument = maria.get_instrument(array=array)

print(instrument)
instrument.plot()

In [ ]:
from maria import Plan

plan = Plan.generate(duration=1800, 
                     sample_rate=25, 
                     start_time="2026-08-05T06:00:00",
                     scan_type="raster",
                     scan_parameters={"n": [(25, 1), (1, 26)], 
                                       "ra_center": 0,
                                       "dec_center": -60,
                                       "x_throw": 2,
                                       "y_throw": 2
                                     },
                     site="cerro_toco")

plan.plot(frames=["az/el", "ra/dec", "glon/glat"])

In [ ]:
sim = maria.Simulation(
    instrument=instrument,
    plans=[plan],
    site="cerro_toco",
    cmb="generate",
    cmb_kwargs={"nside": 256},
)

print(sim)

In [ ]:
sim.maps["cmb"].nu

In [ ]:
instrument.bands[0].nu

In [ ]:
tods = sim.run()
tods[0].plot()

Binning the data gives us a 

In [ ]:
from maria.mapping import *

plot_kwargs = {"slices": dict(stokes=["I", "Q", "U"], nu=[[0], [1]]), "contrast": 1e-2}

bin_mapper = BinMapper(tods=tods,
                       units="uK_CMB",
                       resolution=2 / 60,
                       frame="ra/dec",
                       tod_preprocessing={
                        "remove_polynomial": {"time": 3, "elevation": 3},
                       },
                      )

bin_mapper.run()
bin_mapper.map.plot(**plot_kwargs)

In [ ]:
from maria.mapping.ml_mapper import *

ml_mapper = MaximumLikelihoodMapper(tods=tods,
                                    units="uK_CMB",
                                    resolution=2 / 60,
                                    frame="ra/dec",
                                    tod_preprocessing={
                                    "remove_polynomial": {"time": 3, "elevation": 3},
                                   },
                                   )

In [ ]:
print(ml_mapper.map)
ml_mapper.map.plot(**plot_kwargs)

In [ ]:
ml_mapper.fit(epochs=1, 
              max_steps_per_epoch=50, 
              plot=True, 
              plot_kwargs=plot_kwargs)